# 03 — Treinamento YOLOv8n com dataset validado

Este notebook é necessário somente para um **novo treinamento**. Para continuar com
o modelo atual e a avaliação automática, use o 05d no Windows.

O notebook 02 cria `homeobjects_yolo_validado.zip` automaticamente. O arquivo deve estar disponível
para `Meu Drive/TCC_Visao_Computacional/datasets/`. O Colab verificará o conteúdo contra
o manifesto antes do treinamento. Não há preenchimento manual de métricas ou planilhas.

Cada execução usa um nome novo para preservar o baseline anterior e os resultados do
Drive. Ativar GPU e autorizar o Drive continuam sendo configurações da sessão.


## 1. Ativar GPU no Colab

No Google Colab, acesse:

```text
Ambiente de execução
    → Alterar tipo de ambiente de execução
    → Acelerador de hardware
    → GPU
```

Depois execute a célula abaixo.

In [9]:
import platform
import sys

import torch

print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU não está disponível. "
        "No Colab, ative uma GPU antes de continuar."
    )

Python: 3.13.15
Sistema: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu130
CUDA disponível: True
GPU: Tesla T4


## 2. Instalar Ultralytics

A versão instalada será registrada para manter a rastreabilidade do experimento.

In [10]:
# Se a biblioteca já estiver instalada, a instalação não a atualiza.
!pip install -q ultralytics


In [11]:
import ultralytics

from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)

Ultralytics: 8.4.176


## 3. Montar o Google Drive

O Drive será utilizado para:

- ler o ZIP do dataset;
- salvar pesos do modelo;
- salvar métricas;
- salvar gráficos;
- preservar os resultados mesmo após o encerramento da sessão do Colab.

In [12]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Configurar caminhos e parâmetros do experimento

Para o primeiro baseline usaremos:

- modelo: `yolov8n.pt`;
- resolução: `640 x 640`;
- épocas: `50`;
- batch automático;
- seed: `42`.

Esses valores poderão ser alterados em experimentos futuros, mas o primeiro baseline deve permanecer registrado.

In [13]:
from pathlib import Path
from datetime import datetime

DRIVE_TCC = Path('/content/drive/MyDrive/TCC_Visao_Computacional')
ZIP_DATASET = DRIVE_TCC / 'datasets/homeobjects_yolo_validado.zip'
NOME_EXPERIMENTO = 'yolov8n_automatico_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
PASTA_RESULTADOS_DRIVE = DRIVE_TCC / 'resultados' / NOME_EXPERIMENTO
PASTA_TRABALHO = Path('/content/tcc_visao_computacional') / NOME_EXPERIMENTO
PASTA_DATASET = PASTA_TRABALHO / 'homeobjects_yolo'

MODELO_BASE = 'yolov8n.pt'
EPOCHS = 50
IMGSZ = 640
BATCH = -1
BATCH_AVALIACAO = 16
SEED = 42
PATIENCE = 15
WORKERS = 2
MODO_TESTE_RAPIDO = False
if MODO_TESTE_RAPIDO:
    EPOCHS = 3
PASTA_RESULTADOS_DRIVE.mkdir(parents=True, exist_ok=False)
print('Pacote:', ZIP_DATASET)
print('Experimento:', NOME_EXPERIMENTO)


Pacote: /content/drive/MyDrive/TCC_Visao_Computacional/datasets/homeobjects_yolo_validado.zip
Experimento: yolov8n_automatico_20261013_034255_499777


## 5. Verificar o pacote produzido automaticamente

Use o ZIP gerado pelo notebook 02 revisado, sem alterar seu conteúdo.


In [14]:
if not ZIP_DATASET.is_file():
    raise FileNotFoundError(f'ZIP gerado pelo notebook 02 esperado em: {ZIP_DATASET}')
print(f'Pacote encontrado: {ZIP_DATASET.stat().st_size / 1024**2:.2f} MB')


Pacote encontrado: 389.86 MB


## 6. Extrair o dataset no armazenamento temporário do Colab

O treinamento ocorrerá utilizando o armazenamento local do Colab, que é mais rápido do que ler milhares de imagens diretamente do Google Drive.

In [15]:
import zipfile

PASTA_TRABALHO.mkdir(parents=True, exist_ok=False)
destino = PASTA_TRABALHO.resolve()
with zipfile.ZipFile(ZIP_DATASET) as pacote:
    for membro in pacote.infolist():
        final = (destino / membro.filename).resolve()
        if final != destino and destino not in final.parents:
            raise ValueError('ZIP contém caminho fora da pasta de trabalho.')
    pacote.extractall(destino)
print('Pacote extraído em pasta nova.')


Pacote extraído em pasta nova.


## 7. Localizar automaticamente o `data.yaml`

Essa etapa torna o notebook tolerante ao caso do ZIP conter ou não uma pasta adicional na raiz.

In [16]:
arquivos_yaml = list(PASTA_TRABALHO.rglob("data.yaml"))

if len(arquivos_yaml) == 0:
    raise FileNotFoundError("Nenhum data.yaml foi encontrado após a extração.")

if len(arquivos_yaml) > 1:
    print("Foram encontrados vários data.yaml:")
    for arquivo in arquivos_yaml:
        print(" -", arquivo)

    raise RuntimeError(
        "Há mais de um data.yaml. "
        "Mantenha apenas o dataset homeobjects_yolo no ZIP."
    )

CAMINHO_YAML_ORIGINAL = arquivos_yaml[0]
RAIZ_DATASET = CAMINHO_YAML_ORIGINAL.parent

print("Dataset localizado em:", RAIZ_DATASET)
print("YAML original:", CAMINHO_YAML_ORIGINAL)

Dataset localizado em: /content/tcc_visao_computacional/yolov8n_automatico_20261013_034255_499777/homeobjects_yolo
YAML original: /content/tcc_visao_computacional/yolov8n_automatico_20261013_034255_499777/homeobjects_yolo/data.yaml


## 8. Ajustar o `data.yaml` para o ambiente Colab

O YAML criado no Windows contém um caminho absoluto do computador local.

Aqui criamos uma cópia específica para o Colab, mantendo:

- `train`;
- `val`;
- `test`;
- as 12 classes.

Apenas o campo `path` é atualizado.

In [17]:
import yaml

with CAMINHO_YAML_ORIGINAL.open("r", encoding="utf-8") as arquivo:
    dados_yaml = yaml.safe_load(arquivo)

dados_yaml["path"] = RAIZ_DATASET.as_posix()

CAMINHO_YAML_COLAB = RAIZ_DATASET / "data_colab.yaml"

with CAMINHO_YAML_COLAB.open("w", encoding="utf-8") as arquivo:
    yaml.safe_dump(
        dados_yaml,
        arquivo,
        allow_unicode=True,
        sort_keys=False,
    )

print(CAMINHO_YAML_COLAB.read_text(encoding="utf-8"))

path: /content/tcc_visao_computacional/yolov8n_automatico_20261013_034255_499777/homeobjects_yolo
train: images/train
val: images/val
test: images/test
names:
  0: bed
  1: sofa
  2: chair
  3: table
  4: lamp
  5: tv
  6: laptop
  7: wardrobe
  8: window
  9: door
  10: potted plant
  11: photo frame



## 9. Verificar o manifesto e a estrutura

Esta conferência verifica todos os arquivos empacotados, a contagem por conjunto e o
mapa das classes. O TEST permanece com 403 imagens. Ao usar grupos, as quantidades de
train e val podem variar; o manifesto registra as quantidades realmente preparadas.


In [18]:
import hashlib
import json
import pandas as pd

CAMINHO_MANIFESTO = PASTA_TRABALHO / 'verificacao_preparacao.json'
if not CAMINHO_MANIFESTO.is_file():
    raise RuntimeError('Pacote sem manifesto. Gere o ZIP pelo notebook 02 revisado.')
manifesto_pacote = json.loads(CAMINHO_MANIFESTO.read_text(encoding='utf-8'))
if manifesto_pacote['test']['imagens'] != 403:
    raise RuntimeError('O pacote não corresponde ao TEST de 403 imagens.')
def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with Path(caminho).open('rb') as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()
for item in manifesto_pacote['arquivos']:
    arquivo = (PASTA_TRABALHO / item['arquivo']).resolve()
    if PASTA_TRABALHO.resolve() not in arquivo.parents:
        raise RuntimeError('Manifesto contém caminho inválido.')
    if not arquivo.is_file() or sha256_arquivo(arquivo) != item['sha256']:
        raise RuntimeError(f'Arquivo ausente ou alterado: {item["arquivo"]}')
classes_manifesto = {int(k): v for k, v in manifesto_pacote['classes'].items()}
classes_yaml = dados_yaml['names']
classes_yaml = dict(enumerate(classes_yaml)) if isinstance(classes_yaml, list) else {int(k): v for k, v in classes_yaml.items()}
if classes_yaml != classes_manifesto:
    raise RuntimeError('Mapa de classes diferente do manifesto.')
EXTENSOES_IMAGEM = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
resumo_dataset = []
for split in ['train', 'val', 'test']:
    imagens = [p for p in (RAIZ_DATASET / 'images' / split).rglob('*')
               if p.is_file() and p.suffix.lower() in EXTENSOES_IMAGEM]
    labels = list((RAIZ_DATASET / 'labels' / split).rglob('*.txt'))
    if len(imagens) != len(labels) or len(imagens) != manifesto_pacote['imagens_por_split'].get(split, 0):
        raise RuntimeError(f'Contagem incompatível no conjunto {split}.')
    if not imagens:
        raise RuntimeError(f'Conjunto {split} vazio.')
    resumo_dataset.append({'split': split, 'imagens': len(imagens), 'labels': len(labels)})
df_resumo = pd.DataFrame(resumo_dataset)
display(df_resumo)
print('Pacote verificado automaticamente.')


,split,imagens,labels
0,train,1824,1824
1,val,456,456
2,test,403,403


Pacote verificado automaticamente.


## 10. Registrar informações do experimento

Salvamos os principais parâmetros antes do treinamento para garantir a rastreabilidade metodológica.

In [19]:
import json
from datetime import datetime

informacoes_experimento = {
    "data_execucao_utc": datetime.utcnow().isoformat(),
    "modelo_base": MODELO_BASE,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "batch_avaliacao": BATCH_AVALIACAO,
    "test_sha256": manifesto_pacote["test"]["sha256"],
    "nome_experimento": NOME_EXPERIMENTO,
    "seed": SEED,
    "patience": PATIENCE,
    "workers": WORKERS,
    "pytorch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "dataset": df_resumo.to_dict(orient="records"),
    "classes": dados_yaml["names"],
}

caminho_info = PASTA_RESULTADOS_DRIVE / "configuracao_experimento.json"

with caminho_info.open("w", encoding="utf-8") as arquivo:
    json.dump(
        informacoes_experimento,
        arquivo,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(informacoes_experimento, ensure_ascii=False, indent=2))


/tmp/ipykernel_3481/1012202864.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "data_execucao_utc": datetime.utcnow().isoformat(),


{
  "data_execucao_utc": "2026-10-13T03:43:02.339614",
  "modelo_base": "yolov8n.pt",
  "epochs": 50,
  "imgsz": 640,
  "batch": -1,
  "batch_avaliacao": 16,
  "test_sha256": "f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89",
  "nome_experimento": "yolov8n_automatico_20261013_034255_499777",
  "seed": 42,
  "patience": 15,
  "workers": 2,
  "pytorch": "2.11.0+cu130",
  "ultralytics": "8.4.176",
  "gpu": "Tesla T4",
  "dataset": [
    {
      "split": "train",
      "imagens": 1824,
      "labels": 1824
    },
    {
      "split": "val",
      "imagens": 456,
      "labels": 456
    },
    {
      "split": "test",
      "imagens": 403,
      "labels": 403
    }
  ],
  "classes": {
    "0": "bed",
    "1": "sofa",
    "2": "chair",
    "3": "table",
    "4": "lamp",
    "5": "tv",
    "6": "laptop",
    "7": "wardrobe",
    "8": "window",
    "9": "door",
    "10": "potted plant",
    "11": "photo frame"
  }
}


# 11. Treinamento baseline

Agora iniciamos o fine-tuning do **YOLOv8n pré-treinado**.

O modelo utilizará somente:

```text
train
   +
validation
```

O conjunto `test` continuará reservado.

In [20]:
modelo = YOLO(MODELO_BASE)

resultado_treino = modelo.train(
    data=str(CAMINHO_YAML_COLAB),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    workers=WORKERS,
    device=0,
    pretrained=True,
    optimizer="auto",
    plots=True,
    save=True,
    verbose=True,
    project="/content/experimentos",
    name=NOME_EXPERIMENTO,
    exist_ok=False,
)


Ultralytics 8.4.176 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/tcc_visao_computacional/yolov8n_automatico_20261013_034255_499777/homeobjects_yolo/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, mul

## 12. Localizar o melhor modelo

O Ultralytics salva dois pesos principais:

```text
best.pt
last.pt
```

Para as próximas etapas do TCC utilizaremos principalmente o **`best.pt`**.

In [21]:
PASTA_RUN = Path("/content/experimentos") / NOME_EXPERIMENTO
CAMINHO_BEST = PASTA_RUN / "weights" / "best.pt"
CAMINHO_LAST = PASTA_RUN / "weights" / "last.pt"

if not CAMINHO_BEST.exists():
    raise FileNotFoundError(
        f"O best.pt não foi encontrado em {CAMINHO_BEST}"
    )

print("Best:", CAMINHO_BEST)
print("Last:", CAMINHO_LAST)
print()
print(f"Tamanho best.pt: {CAMINHO_BEST.stat().st_size / 1024**2:.2f} MB")


Best: /content/experimentos/yolov8n_automatico_20261013_034255_499777/weights/best.pt
Last: /content/experimentos/yolov8n_automatico_20261013_034255_499777/weights/last.pt

Tamanho best.pt: 6.19 MB


# 13. Avaliação final no conjunto TEST

Esta é uma etapa importante para o TCC.

O modelo treinado é carregado a partir do `best.pt` e avaliado nas **403 imagens reservadas para teste**.

As principais métricas de interesse serão:

- Precision;
- Recall;
- mAP@50;
- mAP@50-95.

Além disso, o Ultralytics gerará gráficos e matriz de confusão.

In [22]:
melhor_modelo = YOLO(str(CAMINHO_BEST))

resultado_test = melhor_modelo.val(
    data=str(CAMINHO_YAML_COLAB),
    split="test",
    imgsz=IMGSZ,
    batch=BATCH_AVALIACAO,
    device=0,
    plots=True,
    verbose=True,
    project="/content/experimentos",
    name=NOME_EXPERIMENTO + "_test",
    exist_ok=False,
)


Ultralytics 8.4.176 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,135,380 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1889.6±229.5 MB/s, size: 126.0 KB)
val: Scanning /content/tcc_visao_computacional/yolov8n_automatico_20261013_034255_499777/homeobjects_yolo/labels/test... 403 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 403/403 1.8Kit/s 0.2s
val: New cache created: /content/tcc_visao_computacional/yolov8n_automatico_20261013_034255_499777/homeobjects_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 1.9it/s 13.9s
                   all        403       3465      0.715      0.625      0.682      0.468
                   bed         22         22      0.633      0.636      0.708      0.521
                  sofa        285        397       0.85      0.758      0.852      0.642
                 chair 

## 14. Resumo das métricas do TEST

As métricas abaixo serão salvas também em CSV para uso posterior nos capítulos do TCC.

In [23]:
metricas_test = {
    "precision": float(resultado_test.box.mp),
    "recall": float(resultado_test.box.mr),
    "map50": float(resultado_test.box.map50),
    "map50_95": float(resultado_test.box.map),
}

df_metricas_test = pd.DataFrame(
    [
        {"metrica": metrica, "valor": valor}
        for metrica, valor in metricas_test.items()
    ]
)

display(df_metricas_test)

df_metricas_test.to_csv(
    PASTA_RESULTADOS_DRIVE / "metricas_test.csv",
    index=False,
    encoding="utf-8-sig",
)

,metrica,valor
0,precision,0.715118
1,recall,0.624950
2,map50,0.681739
3,map50_95,0.468251


## 15. Métricas por classe

Como o HomeObjects-3K possui forte desbalanceamento entre classes, não devemos avaliar o modelo apenas pelo mAP global.

Esta etapa registra o desempenho individual das 12 classes.

In [24]:
nomes_classes = melhor_modelo.names

metricas_classes = []

for indice, classe_id in enumerate(resultado_test.box.ap_class_index):
    classe_id = int(classe_id)

    metricas_classes.append(
        {
            "id_classe": classe_id,
            "classe": nomes_classes[classe_id],
            "precision": float(resultado_test.box.p[indice]),
            "recall": float(resultado_test.box.r[indice]),
            "map50": float(resultado_test.box.ap50[indice]),
            "map50_95": float(resultado_test.box.ap[indice]),
        }
    )

df_metricas_classes = pd.DataFrame(metricas_classes)

display(
    df_metricas_classes.sort_values(
        "map50_95",
        ascending=False,
    )
)

df_metricas_classes.to_csv(
    PASTA_RESULTADOS_DRIVE / "metricas_por_classe_test.csv",
    index=False,
    encoding="utf-8-sig",
)

,id_classe,classe,precision,recall,map50,map50_95
11,11,photo frame,0.857880,0.735714,0.828426,0.653813
1,1,sofa,0.850013,0.758186,0.851903,0.641980
5,5,tv,0.784700,0.675041,0.741818,0.591898
3,3,table,0.749538,0.717718,0.796067,0.571361
0,0,bed,0.632953,0.636364,0.707934,0.521199
6,6,laptop,0.804005,0.750000,0.782037,0.460111
2,2,chair,0.695550,0.666659,0.680426,0.459273
10,10,potted plant,0.771049,0.634053,0.697501,0.376880
9,9,door,0.553352,0.470588,0.538366,0.372229
7,7,wardrobe,0.647055,0.467890,0.530785,0.356401


## 16. Copiar todos os resultados para o Google Drive

Serão preservados:

- `best.pt`;
- `last.pt`;
- `results.csv`;
- curvas de treinamento;
- matriz de confusão;
- gráficos de Precision/Recall;
- avaliação do TEST;
- configurações do experimento.

In [25]:
import shutil
shutil.copytree(PASTA_RUN, PASTA_RESULTADOS_DRIVE / 'treinamento')
shutil.copytree(Path('/content/experimentos') / (NOME_EXPERIMENTO + '_test'),
                PASTA_RESULTADOS_DRIVE / 'avaliacao_test')
shutil.copy2(CAMINHO_BEST, PASTA_RESULTADOS_DRIVE / 'best.pt')
shutil.copy2(CAMINHO_MANIFESTO, PASTA_RESULTADOS_DRIVE / 'verificacao_preparacao.json')
print('Resultados preservados em pasta nova:', PASTA_RESULTADOS_DRIVE)


Resultados preservados em pasta nova: /content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_automatico_20261013_034255_499777


## 17. Testar o modelo em algumas imagens do TEST

Esta etapa gera exemplos visuais das predições do modelo treinado.

Ela **não altera as métricas**. Serve apenas para inspeção qualitativa.

In [26]:
import random

random.seed(SEED)

pasta_test = RAIZ_DATASET / "images" / "test"

imagens_test = [
    arquivo
    for arquivo in pasta_test.rglob("*")
    if arquivo.is_file()
    and arquivo.suffix.lower() in EXTENSOES_IMAGEM
]

quantidade_amostras = min(8, len(imagens_test))

amostras = random.sample(
    imagens_test,
    quantidade_amostras,
)

resultados_predicao = melhor_modelo.predict(
    source=[str(imagem) for imagem in amostras],
    imgsz=IMGSZ,
    conf=0.25,
    device=0,
    save=True,
    project="/content/experimentos",
    name=NOME_EXPERIMENTO + "_amostras",
    exist_ok=False,
)

print(f"{quantidade_amostras} imagens processadas.")



0: 640x640 3 sofas, 1 table, 1 lamp, 1 tv, 1 wardrobe, 2 windows, 2 potted plants, 6.8ms
1: 640x640 2 sofas, 1 table, 1 wardrobe, 4 potted plants, 1 photo frame, 6.8ms
2: 640x640 1 sofa, 2 chairs, 2 tables, 1 lamp, 6.8ms
3: 640x640 2 chairs, 1 table, 2 potted plants, 10 photo frames, 6.8ms
4: 640x640 1 sofa, 2 chairs, 1 table, 1 lamp, 1 tv, 2 windows, 1 potted plant, 1 photo frame, 6.8ms
5: 640x640 1 sofa, 3 tables, 1 lamp, 3 windows, 1 potted plant, 6.8ms
6: 640x640 1 bed, 2 sofas, 1 lamp, 6 windows, 1 potted plant, 1 photo frame, 6.8ms
7: 640x640 1 bed, 2 chairs, 1 table, 2 potted plants, 6.8ms
Speed: 2.1ms preprocess, 6.8ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/experimentos/yolov8n_automatico_20261013_034255_499777_amostras
8 imagens processadas.


## 18. Salvar exemplos de predição no Drive

In [27]:
shutil.copytree(Path('/content/experimentos') / (NOME_EXPERIMENTO + '_amostras'),
                PASTA_RESULTADOS_DRIVE / 'amostras_test')
print('Exemplos visuais salvos. A escolha das amostras foi automática.')


Exemplos visuais salvos. A escolha das amostras foi automática.


## Resultado e próximo passo

O Drive contém uma pasta nova para este experimento com pesos, métricas, gráficos e
manifesto. O baseline anterior permanece preservado. Para avaliar os alertas em texto,
use o 05d no Windows apontando para o novo modelo e para o mesmo TEST original.

Não ajuste hiperparâmetros a partir do TEST. As versões de PyTorch e Ultralytics são
registradas; para comparações, mantenha a versão do ambiente do experimento de referência.
